# Визуализация landmarks (raw / pp) + диагностика
Интерактивный просмотр JSON из `extract_keypoints.py`, включая postprocess (`*_pp.json`).
Режимы: single frame, grid, compare raw vs pp, таймлайн + jump-to-events.

## A) Setup & Imports

In [1]:
from pathlib import Path
import json
import math
import functools

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from IPython.display import display, clear_output, HTML, Javascript

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)

# Widen output area and avoid auto-scaling images
display(HTML("""
<style>
.output_area img, .jp-OutputArea-output img {
    max-width: none !important;
}
.output_scroll {
    max-height: none !important;
}
.jp-OutputArea-output {
    max-height: none !important;
}
</style>
"""))


## B) Paths & file pickers

In [ ]:
# Basic paths (can be overridden via widgets below)
raw_json_default = Path("outputs/tune_out/_tune_tmp/2bc5cf315556") / "2392e0d5-c20d-4497-b9cf-e057f3961645.json"
video_default = Path("test") / "2392e0d5-c20d-4497-b9cf-e057f3961645.mp4"


def find_pp_path(json_path: Path) -> Path | None:
    if not json_path:
        return None
    p = Path(json_path)
    if p.suffix.lower() != ".json":
        return None
    pp = p.with_name(p.stem + "_pp.json")
    return pp if pp.exists() else None


def _pick_json_in_run_dir(run_dir: Path) -> Path | None:
    if not run_dir or not run_dir.exists():
        return None
    candidates = [
        p for p in run_dir.glob("*.json")
        if p.name not in ("manifest.json", "eval_report.json")
    ]
    if not candidates:
        return None
    raw = [p for p in candidates if not p.name.endswith("_pp.json")]
    if raw:
        return sorted(raw)[0]
    return sorted(candidates)[0]


def guess_video_path(meta: dict, json_path: Path) -> Path | None:
    # Try meta hints, then same stem.mp4 next to JSON
    if isinstance(meta, dict):
        for key in ("video", "input", "input_file", "source_file"):
            candidate = meta.get(key)
            if candidate:
                p = Path(candidate)
                if not p.is_absolute() and json_path:
                    p = Path(json_path).parent / p
                if p.exists():
                    return p
    if json_path:
        p = Path(json_path)
        mp4 = p.with_suffix(".mp4")
        if mp4.exists():
            return mp4
    return None


def load_run(json_path: Path) -> tuple[dict, list]:
    if not json_path:
        return {}, []
    p = Path(json_path)
    if not p.exists():
        return {}, []
    data = json.loads(p.read_text(encoding="utf-8"))
    return data.get("meta", {}), data.get("frames", [])


# Widget controls (graceful fallback if ipywidgets missing)
if HAS_WIDGETS:
    json_path_widget = widgets.Text(value=str(raw_json_default), description="raw json", layout=widgets.Layout(width="700px"))
    use_pp_widget = widgets.Checkbox(value=True, description="use_pp")
    compare_widget = widgets.Checkbox(value=False, description="compare_raw_vs_pp")
    video_path_widget = widgets.Text(value=str(video_default), description="video", layout=widgets.Layout(width="700px"))
    tune_root_widget = widgets.Text(value="", description="top_runs", layout=widgets.Layout(width="700px"))
    tune_select_widget = widgets.Dropdown(options=[""], description="run_dir", layout=widgets.Layout(width="600px"))

    def _refresh_runs(_=None):
        root = Path(tune_root_widget.value) if tune_root_widget.value else None
        opts = [""]
        if root and root.exists():
            opts += [str(p) for p in sorted(root.iterdir()) if p.is_dir()]
        tune_select_widget.options = opts

    def _on_run_select(change):
        if not change.get("new"):
            return
        run_dir = Path(change["new"])
        pick = _pick_json_in_run_dir(run_dir)
        if pick and json_path_widget:
            json_path_widget.value = str(pick)

    tune_root_widget.observe(_refresh_runs, "value")
    tune_select_widget.observe(_on_run_select, "value")
    _refresh_runs()

    display(widgets.VBox([
        widgets.HBox([json_path_widget, use_pp_widget, compare_widget]),
        video_path_widget,
        widgets.HBox([tune_root_widget, tune_select_widget]),
    ]))
else:
    json_path_widget = None
    use_pp_widget = None
    compare_widget = None
    video_path_widget = None
    tune_root_widget = None
    tune_select_widget = None
    print("ipywidgets not available; using defaults.")


## C) Load & normalize frame schema

In [3]:
# Unified access helpers (support both 'hand 1' and 'hand_1' style keys)

def g(fr: dict, *keys, default=None):
    for k in keys:
        if k in fr:
            return fr[k]
    return default


def _hand_keys(idx: int):
    return {
        "pts": (f"hand {idx}", f"hand_{idx}"),
        "score": (f"hand {idx}_score", f"hand_{idx}_score"),
        "score_gate": (f"hand {idx}_score_gate", f"hand_{idx}_score_gate"),
        "source": (f"hand {idx}_source", f"hand_{idx}_source"),
        "state": (f"hand {idx}_state", f"hand_{idx}_state"),
        "anchor": (f"hand {idx}_is_anchor", f"hand_{idx}_is_anchor"),
        "reject": (f"hand {idx}_reject_reason", f"hand_{idx}_reject_reason"),
        "pose_quality": (f"hand {idx}_pose_quality", f"hand_{idx}_pose_quality"),
        "wrist_z": (f"hand {idx}_wrist_z", f"hand_{idx}_wrist_z"),
        "sanity_scale_ratio": (f"hand {idx}_sanity_scale_ratio", f"hand_{idx}_sanity_scale_ratio"),
        "sanity_wrist_jump": (f"hand {idx}_sanity_wrist_jump", f"hand_{idx}_sanity_wrist_jump"),
        "sanity_bone_max_err": (f"hand {idx}_sanity_bone_max_err", f"hand_{idx}_sanity_bone_max_err"),
        "sanity_bone_worst": (f"hand {idx}_sanity_bone_worst", f"hand_{idx}_sanity_bone_worst"),
        "sanity_stage": (f"hand {idx}_sanity_stage", f"hand_{idx}_sanity_stage"),
        "side_ok": (f"side_ok_{idx}",),
        "track_ok": (f"hand_{idx}_track_ok", f"hand {idx}_track_ok"),
        "track_age": (f"hand_{idx}_track_age", f"hand {idx}_track_age"),
        "track_reset": (f"hand_{idx}_track_reset", f"hand {idx}_track_reset"),
        "tracker_ready": (f"hand_{idx}_tracker_ready", f"hand {idx}_tracker_ready"),
        "tracker_last_score": (f"hand {idx}_tracker_last_score", f"hand_{idx}_tracker_last_score"),
        "tracker_last_ts": (f"hand {idx}_tracker_last_ts", f"hand_{idx}_tracker_last_ts"),
        "sp_attempt": (f"hand_{idx}_sp_attempted", f"hand {idx}_sp_attempted"),
        "sp_recovered": (f"hand_{idx}_sp_recovered", f"hand {idx}_sp_recovered"),
        "sp_roi": (f"hand {idx}_sp_roi_px", f"hand_{idx}_sp_roi_px"),
        "sp_hint": (f"hand {idx}_sp_center_hint_px", f"hand_{idx}_sp_center_hint_px"),
        "sp_debug": (f"hand {idx}_sp_debug", f"hand_{idx}_sp_debug"),
        "sp_params": (f"hand {idx}_sp_params", f"hand_{idx}_sp_params"),
        "pp_applied": (f"hand {idx}_pp_applied", f"hand_{idx}_pp_applied"),
        "pp_reason": (f"hand {idx}_pp_reason", f"hand_{idx}_pp_reason"),
        "pp_overrode": (f"hand {idx}_pp_overrode", f"hand_{idx}_pp_overrode"),
        "pp_overrode_reason": (f"hand {idx}_pp_overrode_reason", f"hand_{idx}_pp_overrode_reason"),
    }


def get_hand(fr: dict, idx: int) -> dict:
    keys = _hand_keys(idx)
    return {
        "pts": g(fr, *keys["pts"], default=None),
        "score": g(fr, *keys["score"], default=None),
        "score_gate": g(fr, *keys["score_gate"], default=None),
        "source": g(fr, *keys["source"], default=None),
        "state": g(fr, *keys["state"], default=None),
        "anchor": g(fr, *keys["anchor"], default=False),
        "reject": g(fr, *keys["reject"], default=None),
        "pose_quality": g(fr, *keys["pose_quality"], default=None),
        "wrist_z": g(fr, *keys["wrist_z"], default=None),
        "sanity_scale_ratio": g(fr, *keys["sanity_scale_ratio"], default=None),
        "sanity_wrist_jump": g(fr, *keys["sanity_wrist_jump"], default=None),
        "sanity_bone_max_err": g(fr, *keys["sanity_bone_max_err"], default=None),
        "sanity_bone_worst": g(fr, *keys["sanity_bone_worst"], default=None),
        "sanity_stage": g(fr, *keys["sanity_stage"], default=None),
        "side_ok": g(fr, *keys["side_ok"], default=None),
        "track_ok": g(fr, *keys["track_ok"], default=False),
        "track_age": g(fr, *keys["track_age"], default=None),
        "track_reset": g(fr, *keys["track_reset"], default=False),
        "tracker_ready": g(fr, *keys["tracker_ready"], default=None),
        "tracker_last_score": g(fr, *keys["tracker_last_score"], default=None),
        "tracker_last_ts": g(fr, *keys["tracker_last_ts"], default=None),
        "sp_attempt": g(fr, *keys["sp_attempt"], default=False),
        "sp_recovered": g(fr, *keys["sp_recovered"], default=False),
        "sp_roi": g(fr, *keys["sp_roi"], default=None),
        "sp_hint": g(fr, *keys["sp_hint"], default=None),
        "sp_debug": g(fr, *keys["sp_debug"], default=None),
        "sp_params": g(fr, *keys["sp_params"], default=None),
        "pp_applied": g(fr, *keys["pp_applied"], default=False),
        "pp_reason": g(fr, *keys["pp_reason"], default=None),
        "pp_overrode": g(fr, *keys["pp_overrode"], default=False),
        "pp_overrode_reason": g(fr, *keys["pp_overrode_reason"], default=None),
        "occluded": g(fr, f"occluded_{idx}", default=False),
        "occluded_ttl": g(fr, f"occluded_{idx}_ttl", default=None),
        "occlusion_saved": g(fr, f"occlusion_saved_{idx}", default=False),
        "missing_pre_occ": g(fr, f"missing_pre_occ_{idx}", default=False),
        "occlusion_freeze_age": g(fr, f"occlusion_freeze_age_{idx}", default=None),
        "occlusion_freeze_max": g(fr, "occlusion_freeze_max", default=None),
    }

def get_pose(fr: dict):
    return g(fr, "pose", "pose_3d", default=None)


def get_hand_gate(meta: dict):
    gate = {}
    if isinstance(meta, dict):
        gate = meta.get("hand_score_gate") or {}
    lo = gate.get("lo", None)
    hi = gate.get("hi", None)
    return lo, hi, gate.get("min_hand_score_legacy", None)


def is_sanity_reject(reason: str | None) -> bool:
    if not reason:
        return False
    return "sanity" in str(reason).lower()


def _has_reason(reason: str | None, token: str) -> bool:
    if not reason:
        return False
    return token in str(reason)




def _hand_pts(fr, idx):
    return g(fr, f"hand {idx}", f"hand_{idx}", default=None)

def _pp_flag(fr, idx, suffix):
    return bool(g(fr, f"hand {idx}_{suffix}", f"hand_{idx}_{suffix}", default=False))

def pp_diff_indices(frames_raw, frames_pp):
    if not frames_raw or not frames_pp:
        return []
    diffs = []
    for i, (r, p) in enumerate(zip(frames_raw, frames_pp)):
        r1 = _hand_pts(r, 1)
        r2 = _hand_pts(r, 2)
        p1 = _hand_pts(p, 1)
        p2 = _hand_pts(p, 2)
        if r1 != p1 or r2 != p2:
            diffs.append(i)
            continue
        if _pp_flag(p, 1, "pp_applied") or _pp_flag(p, 2, "pp_applied"):
            diffs.append(i)
            continue
        if _pp_flag(p, 1, "pp_overrode") or _pp_flag(p, 2, "pp_overrode"):
            diffs.append(i)
            continue
    return diffs

def pp_diff_stats(frames_raw, frames_pp):
    if not frames_raw or not frames_pp:
        return {"diff_frames": 0, "pp_applied": 0, "pp_overrode": 0}
    diffs = pp_diff_indices(frames_raw, frames_pp)
    pp_applied = 0
    pp_overrode = 0
    for p in frames_pp:
        if _pp_flag(p, 1, "pp_applied") or _pp_flag(p, 2, "pp_applied"):
            pp_applied += 1
        if _pp_flag(p, 1, "pp_overrode") or _pp_flag(p, 2, "pp_overrode"):
            pp_overrode += 1
    return {"diff_frames": len(diffs), "pp_applied": pp_applied, "pp_overrode": pp_overrode}
def load_all_runs():
    json_path = Path(json_path_widget.value) if json_path_widget else raw_json_default
    if tune_select_widget and tune_select_widget.value:
        run_dir = Path(tune_select_widget.value)
        pick = _pick_json_in_run_dir(run_dir)
        if pick:
            json_path = pick

    pp_hint = None
    if json_path and json_path.name.endswith("_pp.json"):
        base_stem = json_path.stem[:-3] if json_path.stem.endswith("_pp") else json_path.stem
        raw_candidate = json_path.with_name(base_stem + ".json")
        if raw_candidate.exists():
            pp_hint = json_path
            json_path = raw_candidate

    meta_raw, frames_raw = load_run(json_path)
    pp_path = pp_hint or find_pp_path(json_path)
    meta_pp = None
    frames_pp = None
    if (use_pp_widget.value if use_pp_widget else True) and pp_path is not None and pp_path.exists():
        meta_pp, frames_pp = load_run(pp_path)
    return {
        "raw": {"path": json_path, "meta": meta_raw, "frames": frames_raw},
        "pp": {"path": pp_path, "meta": meta_pp, "frames": frames_pp} if frames_pp is not None else None,
    }


RUNS = load_all_runs()


## D) Drawing overlay

In [4]:
from collections import OrderedDict

HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
]

POSE_EDGES = [
    (11,13),(13,15),
    (12,14),(14,16),
    (11,12),
    (23,24),
    (11,23),(12,24),
    (0,9),(0,10),
]


def _is_norm_xy(points):
    if not points:
        return False
    xs = [p.get("x", 0.0) for p in points]
    ys = [p.get("y", 0.0) for p in points]
    return max(xs) <= 2.0 and max(ys) <= 2.0


def _to_px(points, W, H):
    if not points:
        return None
    if _is_norm_xy(points):
        return [{"x": p["x"] * W, "y": p["y"] * H, "z": p.get("z", 0.0)} for p in points]
    return points


def _bbox(pts):
    xs = [p["x"] for p in pts]
    ys = [p["y"] for p in pts]
    return min(xs), min(ys), max(xs), max(ys)


def _draw_text(img, lines, origin=(10, 20), line_h=18, font_scale=0.5, color=(255, 255, 255)):
    x, y = origin
    for line in lines:
        cv2.putText(img, line, (x, y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, color, 1, cv2.LINE_AA)
        y += line_h


def _draw_cross(img, x, y, size=6, color=(0, 255, 255), thickness=1):
    x = int(x); y = int(y)
    cv2.line(img, (x - size, y), (x + size, y), color, thickness)
    cv2.line(img, (x, y - size), (x, y + size), color, thickness)


def _draw_roi(img, roi, color=(255, 255, 0), thickness=1, label=None):
    if not roi or len(roi) < 4:
        return
    x0, y0, x1, y1 = [int(v) for v in roi]
    cv2.rectangle(img, (x0, y0), (x1, y1), color, thickness)
    if label:
        cv2.putText(img, label, (x0 + 4, y0 + 14), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)


def _wrist_xy(pts_px):
    if not pts_px:
        return None
    if len(pts_px) < 1:
        return None
    return float(pts_px[0]["x"]), float(pts_px[0]["y"])


def _style_for_hand(src, state, idx):
    # Colors in BGR
    base = (0, 200, 0) if idx == 1 else (255, 140, 0)
    color = base
    alpha = 0.95
    thickness = 2
    dashed = False
    label = None

    if src in ("tracked",):
        color = (255, 80, 80)
        alpha = 0.85
        thickness = 2
        dashed = True
        label = "tracked"
    elif src in ("occluded",) or state == "occluded":
        color = (0, 0, 255)
        alpha = 0.9
        thickness = 3
        dashed = True
        label = "occluded"
    elif src in ("hold",):
        color = (180, 0, 180)
        alpha = 0.8
        thickness = 2
        dashed = True
        label = "hold"
    elif src in ("interp",):
        color = (0, 255, 255)
        alpha = 0.8
        thickness = 2
        dashed = True
        label = "interp"

    if state == "predicted":
        alpha = min(alpha, 0.8)

    return color, alpha, thickness, dashed, label




def _draw_pose_points(overlay, pose_px, color=(180, 180, 255)):
    if not pose_px:
        return
    for p in pose_px:
        cv2.circle(overlay, (int(p["x"]), int(p["y"])), 2, color, -1)

def _draw_hand(overlay, pts_px, color, thickness=2, dashed=False):
    if not pts_px:
        return
    for k, (i, j) in enumerate(HAND_EDGES):
        if i < len(pts_px) and j < len(pts_px):
            if dashed and (k % 2 == 1):
                continue
            pi, pj = pts_px[i], pts_px[j]
            cv2.line(overlay, (int(pi["x"]), int(pi["y"])), (int(pj["x"]), int(pj["y"])), color, thickness)
    radius = 2 if thickness > 1 else 1
    for p in pts_px:
        cv2.circle(overlay, (int(p["x"]), int(p["y"])), radius, color, -1)


def _blend(base, overlay, alpha=0.6):
    return cv2.addWeighted(overlay, alpha, base, 1 - alpha, 0)


class VideoCache:
    def __init__(self, path: Path | None):
        self.path = Path(path) if path else None
        self.cap = None
        if self.path and self.path.exists():
            self.cap = cv2.VideoCapture(str(self.path))

    def read(self, idx: int):
        if not self.cap:
            return None
        self.cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = self.cap.read()
        if not ok:
            return None
        return frame


class LRUCache:
    def __init__(self, maxsize=256):
        self.maxsize = maxsize
        self.data = OrderedDict()

    def get(self, key):
        if key not in self.data:
            return None
        self.data.move_to_end(key)
        return self.data[key]

    def set(self, key, value):
        self.data[key] = value
        self.data.move_to_end(key)
        if len(self.data) > self.maxsize:
            self.data.popitem(last=False)

    def clear(self):
        self.data.clear()




def show_image(img, title=None, zoom=1.0):
    if img is None:
        return
    if zoom is None:
        zoom = 1.0
    try:
        zoom = float(zoom)
    except Exception:
        zoom = 1.0
    if zoom != 1.0:
        h, w = img.shape[:2]
        new_w = max(1, int(w * zoom))
        new_h = max(1, int(h * zoom))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    h, w = img.shape[:2]
    fig_w = min(12.0, max(4.0, w / 120.0))
    fig_h = min(8.0, max(3.0, h / 120.0))
    plt.figure(figsize=(fig_w, fig_h))
    if title:
        plt.title(title)
    plt.imshow(img)
    plt.axis("off")
    plt.show()
def _blank_canvas(meta: dict, fallback=(1280, 720)):
    size = meta.get("size_proc") or meta.get("size_src") or fallback
    if isinstance(size, list) and len(size) >= 2:
        w, h = int(size[0]), int(size[1])
    else:
        w, h = fallback
    return np.zeros((h, w, 3), dtype=np.uint8)


def _anchor_label_color(idx):
    return (0, 220, 0) if idx == 1 else (255, 160, 0)


RENDER_CACHE = LRUCache(maxsize=256)
VIDEO_CACHE = None


def render_frame(frame_idx, *, run="raw", show_pose=True, show_hands=True, show_sp=True, show_text=True, verbose=False):
    if run not in RUNS or RUNS.get(run) is None:
        return None
    frames = RUNS[run]["frames"]
    meta = RUNS[run]["meta"]
    pose_qual_min = None
    if isinstance(meta, dict):
        gate_meta = meta.get("hand_score_gate") or {}
        pose_qual_min = gate_meta.get("pose_dist_qual_min")
    if not (0 <= frame_idx < len(frames)):
        return None

    cache_key = (run, frame_idx, show_pose, show_hands, show_sp, show_text, verbose)
    cached = RENDER_CACHE.get(cache_key)
    if cached is not None:
        return cached

    fr = frames[frame_idx]

    img = None
    if VIDEO_CACHE is not None:
        img = VIDEO_CACHE.read(frame_idx)
    if img is None:
        img = _blank_canvas(meta)

    H, W = img.shape[:2]
    overlay = img.copy()

    # pose
    if show_pose:
        pose = get_pose(fr)
        if pose:
            pose_px = _to_px(pose, W, H)
            if pose_px:
                # draw edges only if pose is full-size
                if len(pose_px) >= 25:
                    for i, j in POSE_EDGES:
                        if i < len(pose_px) and j < len(pose_px):
                            pi, pj = pose_px[i], pose_px[j]
                            cv2.line(overlay, (int(pi["x"]), int(pi["y"])), (int(pj["x"]), int(pj["y"])), (200, 200, 200), 1)
                # always draw points (even if short pose)
                _draw_pose_points(overlay, pose_px)

    if show_pose:
        img = overlay
        overlay = img.copy()

    # hands
    if show_hands:
        for idx in (1, 2):
            h = get_hand(fr, idx)
            pts = h["pts"]
            if pts:
                pts_px = _to_px(pts, W, H)
                color, alpha, thick, dashed, label = _style_for_hand(h["source"], h["state"], idx)
                if pts_px:
                    _draw_hand(overlay, pts_px, color, thickness=thick, dashed=dashed)
                    if h["anchor"]:
                        x0, y0, x1, y1 = _bbox(pts_px)
                        cv2.rectangle(overlay, (int(x0), int(y0)), (int(x1), int(y1)), _anchor_label_color(idx), 2)
                        cv2.putText(overlay, "ANCHOR", (int(x0), max(0, int(y0) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, _anchor_label_color(idx), 1)

                    # label near wrist
                    label_parts = []
                    if label:
                        label_parts.append(label)
                    if label == "tracked" and h.get("track_age") is not None:
                        label_parts[-1] = f"tracked age={h['track_age']}"
                    if label == "occluded" and h.get("occluded_ttl") is not None:
                        label_parts[-1] = f"occluded ttl={h['occluded_ttl']}"
                    if label_parts:
                        wxy = _wrist_xy(pts_px)
                        if wxy:
                            lx, ly = int(wxy[0]) + 4, int(wxy[1]) - 4
                            cv2.putText(overlay, " ".join(label_parts), (lx, ly), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)

                img = _blend(img, overlay, alpha=alpha)
                overlay = img.copy()

    # second-pass overlays
    if show_sp:
        for idx in (1, 2):
            h = get_hand(fr, idx)
            if h["sp_roi"] is not None:
                _draw_roi(img, h["sp_roi"], color=(255, 255, 0), thickness=1, label=f"ROI{idx}")
            if h["sp_hint"] is not None:
                hx, hy = h["sp_hint"]
                _draw_cross(img, hx, hy, size=6, color=(0, 255, 255), thickness=1)
            sp_dbg = h["sp_debug"] or {}
            sel_roi = sp_dbg.get("selected_roi_px")
            if sel_roi:
                _draw_roi(img, sel_roi, color=(0, 255, 255), thickness=2, label=f"SEL{idx}")

        # diagnostic text
    if show_text:
        ts = g(fr, "ts", default=None)
        dt = g(fr, "dt", "dt_ms", default=None)
        overlap_sp = g(fr, "sp_overlap_iou", default=None)
        overlap_occ = g(fr, "occlusion_iou", default=None)
        overlap_hand = g(fr, "iou_hands", default=None)
        overlap_guard_iou = g(fr, "overlap_iou_guard", default=None)
        overlap_guard = g(fr, "overlap_guard", default=None)
        overlap_guard_pre = g(fr, "overlap_guard_pre", default=None)
        overlap_guard_src = g(fr, "overlap_iou_guard_src", default=None)
        overlap_freeze_reason = g(fr, "overlap_freeze_reason", default=None)
        overlap_freeze_side = g(fr, "overlap_freeze_side", default=None)
        overlap_freeze = g(fr, "overlap_freeze", default=None)
        overlap_hand_z = g(fr, "overlap_hand_z_diff", default=None)
        pose_amb = g(fr, "pose_side_ambiguous", default=None)
        swap = g(fr, "swap_applied", default=False)
        dedup = g(fr, "dedup_triggered", default=False)
        gate_lo, gate_hi, gate_min = get_hand_gate(meta)
        gate_str = ""
        if gate_lo is not None or gate_hi is not None:
            gate_str = f" gate=({gate_lo},{gate_hi})"
        if pose_qual_min is not None:
            gate_str += f" pose_q_min={_fmt(pose_qual_min, nd=2)}"
        lines = [f"idx={frame_idx} ts={ts} dt={dt} sp_iou={overlap_sp} occ_iou={overlap_occ} hand_iou={overlap_hand} guard_iou={overlap_guard_iou} guard={overlap_guard}({overlap_guard_src}) pre={overlap_guard_pre} freeze={overlap_freeze}({overlap_freeze_reason}:{overlap_freeze_side}) hand_z={_fmt(overlap_hand_z, nd=4)} amb={pose_amb} swap={swap} dedup={dedup}{gate_str}"]
        for idx in (1, 2):
            h = get_hand(fr, idx)
            occ = h.get("occluded", False)
            occ_ttl = h.get("occluded_ttl", None)
            rej = h["reject"]
            if rej and not verbose:
                rej = str(rej)[:80]
            line = f"H{idx} {h['state']}/{h['source']} score={h['score']} anchor={bool(h['anchor'])}"
            pq = h.get("pose_quality")
            if pq is not None:
                line += f" pose_q={_fmt(pq, nd=3)}"
            if verbose and h.get("side_ok") is not None:
                line += f" side_ok={h.get('side_ok')}"
            if gate_lo is not None or gate_hi is not None:
                line += f" lo/hi={gate_lo}/{gate_hi}"
            if h["track_age"] is not None:
                line += f" track_age={h['track_age']}"
            if occ_ttl is not None:
                line += f" occ_ttl={occ_ttl}"
            freeze_age = h.get("occlusion_freeze_age")
            freeze_max = h.get("occlusion_freeze_max")
            if freeze_age is not None:
                if freeze_max is not None:
                    line += f" occ_freeze={freeze_age}/{freeze_max}"
                else:
                    line += f" occ_freeze={freeze_age}"
            if occ:
                line += " occluded"
            if rej:
                line += f" reject={rej}"
            if h["pp_applied"]:
                line += f" pp={h['pp_reason']}"
            if h["pp_overrode"]:
                line += " pp_overrode"
            lines.append(line)
        if not verbose:
            lines = lines[:6]
        _draw_text(img, lines, origin=(10, 20), line_h=18, font_scale=0.5)

    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    RENDER_CACHE.set(cache_key, rgb)
    return rgb


def _mean_l2_xy(a, b):
    if not a or not b:
        return None
    L = min(len(a), len(b))
    if L <= 0:
        return None
    s = 0.0
    for j in range(L):
        ax, ay = float(a[j]["x"]), float(a[j]["y"])
        bx, by = float(b[j]["x"]), float(b[j]["y"])
        s += math.hypot(ax - bx, ay - by)
    return s / float(L)


def compare_diff(fr_raw, fr_pp):
    rows = []
    for idx in (1, 2):
        h_raw = get_hand(fr_raw, idx)
        h_pp = get_hand(fr_pp, idx)
        raw_pts = h_raw["pts"]
        pp_pts = h_pp["pts"]
        wrist_delta = None
        if raw_pts and pp_pts:
            wrist_delta = math.hypot(raw_pts[0]["x"] - pp_pts[0]["x"], raw_pts[0]["y"] - pp_pts[0]["y"])
        mean_delta = _mean_l2_xy(raw_pts, pp_pts)

        changed = []
        for key in ("source", "state", "anchor", "pp_applied", "pp_overrode"):
            if h_raw.get(key) != h_pp.get(key):
                changed.append(f"{key}:{h_raw.get(key)}->{h_pp.get(key)}")
        raw_score = h_raw.get("score")
        pp_score = h_pp.get("score")
        if raw_score is not None or pp_score is not None:
            try:
                raw_score = None if raw_score is None else round(float(raw_score), 3)
                pp_score = None if pp_score is None else round(float(pp_score), 3)
            except Exception:
                pass
            if raw_score != pp_score:
                changed.append(f"score:{raw_score}->{pp_score}")
        if h_raw.get("reject") != h_pp.get("reject"):
            changed.append(f"reject:{h_raw.get('reject')}->{h_pp.get('reject')}")

        rows.append({
            "hand": idx,
            "wrist_delta": None if wrist_delta is None else round(wrist_delta, 4),
            "mean_kp_delta": None if mean_delta is None else round(mean_delta, 4),
            "changed_fields": "; ".join(changed) if changed else "",
        })
    return pd.DataFrame(rows)




# Frame summary table for quick inspection
def _fmt(v, nd=4):
    if v is None:
        return None
    try:
        if isinstance(v, float):
            return round(v, nd)
        return v
    except Exception:
        return v


def frame_summary_table(fr, idx=None):
    h1 = get_hand(fr, 1)
    h2 = get_hand(fr, 2)
    rows = []
    for hand_idx, h in ((1, h1), (2, h2)):
        rows.append({
            "hand": hand_idx,
            "state": h.get("state"),
            "source": h.get("source"),
            "score": _fmt(h.get("score")),
            "score_gate": _fmt(h.get("score_gate"), nd=3),
            "pose_q": _fmt(h.get("pose_quality"), nd=3),
            "side_ok": h.get("side_ok"),
            "anchor": bool(h.get("anchor")),
            "reject": h.get("reject"),
            "occluded": bool(h.get("occluded")),
            "occl_ttl": h.get("occluded_ttl"),
            "occl_freeze_age": h.get("occlusion_freeze_age"),
            "occl_freeze_max": h.get("occlusion_freeze_max"),
            "occl_saved": bool(h.get("occlusion_saved")),
            "missing_pre": bool(h.get("missing_pre_occ")),
            "track_age": h.get("track_age"),
            "track_ok": bool(h.get("track_ok")),
            "sp_attempt": bool(h.get("sp_attempt")),
            "sp_recovered": bool(h.get("sp_recovered")),
            "pp_applied": bool(h.get("pp_applied")),
            "pp_overrode": bool(h.get("pp_overrode")),
            "pp_reason": h.get("pp_reason"),
        })

    header = {
        "idx": idx if idx is not None else g(fr, "idx", "i", "frame", default=None),
        "ts": _fmt(g(fr, "ts", default=None)),
        "dt": _fmt(g(fr, "dt", "dt_ms", default=None)),
        "swap": bool(g(fr, "swap_applied", default=False)),
        "dedup": bool(g(fr, "dedup_triggered", default=False)),
        "sp_overlap_iou": _fmt(g(fr, "sp_overlap_iou", default=None)),
        "occlusion_iou": _fmt(g(fr, "occlusion_iou", default=None)),
        "iou_hands": _fmt(g(fr, "iou_hands", default=None)),
        "overlap_iou_guard": _fmt(g(fr, "overlap_iou_guard", default=None)),
        "overlap_guard_pre": bool(g(fr, "overlap_guard_pre", default=False)),
        "overlap_guard": bool(g(fr, "overlap_guard", default=False)),
        "overlap_freeze": bool(g(fr, "overlap_freeze", default=False)),
        "overlap_freeze_reason": g(fr, "overlap_freeze_reason", default=None),
        "overlap_freeze_side": g(fr, "overlap_freeze_side", default=None),
        "overlap_guard_src": g(fr, "overlap_iou_guard_src", default=None),
        "overlap_z_hint": _fmt(g(fr, "overlap_z_hint", default=None)),
        "overlap_z_abs": _fmt(g(fr, "overlap_z_abs", default=None)),
        "overlap_hand_z_diff": _fmt(g(fr, "overlap_hand_z_diff", default=None)),
        "occlusion_freeze_max": g(fr, "occlusion_freeze_max", default=None),
        "pose_amb": bool(g(fr, "pose_side_ambiguous", default=False)),
        "pose_side_ratio": _fmt(g(fr, "pose_side_ratio", default=None)),
        "side_cost_current": _fmt(g(fr, "side_cost_current", default=None)),
        "side_cost_swap": _fmt(g(fr, "side_cost_swap", default=None)),
        "side_pref_left": g(fr, "side_pref_left", default=None),
        "side_pref_right": g(fr, "side_pref_right", default=None),
        "sp_overlap": bool(g(fr, "sp_overlap", default=False)),
        "pose_len": len(get_pose(fr) or []),
    }
    return pd.DataFrame(rows), header


def frame_config_table(fr):
    return pd.DataFrame([
        {
            "score_source": g(fr, "score_source", default=None),
            "min_hand_score": _fmt(g(fr, "min_hand_score", default=None), nd=3),
            "hand_score_lo": _fmt(g(fr, "hand_score_lo", default=None), nd=3),
            "hand_score_hi": _fmt(g(fr, "hand_score_hi", default=None), nd=3),
            "anchor_score": _fmt(g(fr, "anchor_score", default=None), nd=3),
            "tracker_init_score": _fmt(g(fr, "tracker_init_score", default=None), nd=3),
            "tracker_update_score": _fmt(g(fr, "tracker_update_score", default=None), nd=3),
            "pose_dist_qual_min": _fmt(g(fr, "pose_dist_qual_min", default=None), nd=3),
        }
    ])


def decision_trace_table(fr):
    h1 = get_hand(fr, 1)
    h2 = get_hand(fr, 2)
    rows = []

    def add(metric, v1, v2, nd=3):
        rows.append({
            "metric": metric,
            "hand1": _fmt(v1, nd=nd),
            "hand2": _fmt(v2, nd=nd),
        })

    add("score_gate", h1.get("score_gate"), h2.get("score_gate"), nd=3)
    add("pose_q", h1.get("pose_quality"), h2.get("pose_quality"), nd=3)
    add("pose_wrist_dist", g(fr, "pose_wrist_dist_1", default=None), g(fr, "pose_wrist_dist_2", default=None), nd=4)
    add("pose_wrist_z_world", g(fr, "pose_wrist_z_left_world", default=None), g(fr, "pose_wrist_z_right_world", default=None), nd=4)
    add("pose_wrist_z_img", g(fr, "pose_wrist_z_left_img", default=None), g(fr, "pose_wrist_z_right_img", default=None), nd=4)
    add("hand_wrist_z", h1.get("wrist_z"), h2.get("wrist_z"), nd=4)
    add("occl_freeze_age", h1.get("occlusion_freeze_age"), h2.get("occlusion_freeze_age"), nd=0)
    add("overlap_hand_z_diff", g(fr, "overlap_hand_z_diff", default=None), None, nd=4)
    add("side_ok", h1.get("side_ok"), h2.get("side_ok"), nd=0)
    add("sanity_scale_ratio", h1.get("sanity_scale_ratio"), h2.get("sanity_scale_ratio"), nd=3)
    add("sanity_wrist_jump", h1.get("sanity_wrist_jump"), h2.get("sanity_wrist_jump"), nd=3)
    add("sanity_bone_max_err", h1.get("sanity_bone_max_err"), h2.get("sanity_bone_max_err"), nd=3)
    add("sanity_bone_worst", h1.get("sanity_bone_worst"), h2.get("sanity_bone_worst"), nd=0)
    add("sanity_stage", h1.get("sanity_stage"), h2.get("sanity_stage"), nd=0)
    add("track_reset", h1.get("track_reset"), h2.get("track_reset"), nd=0)
    add("tracker_last_score", h1.get("tracker_last_score"), h2.get("tracker_last_score"), nd=3)
    add("tracker_last_ts", h1.get("tracker_last_ts"), h2.get("tracker_last_ts"), nd=3)

    return pd.DataFrame(rows)


def frame_debug_table(fr):
    items = [
        ("sp_overlap_iou", g(fr, "sp_overlap_iou", default=None)),
        ("occlusion_iou", g(fr, "occlusion_iou", default=None)),
        ("iou_hands", g(fr, "iou_hands", default=None)),
        ("overlap_iou_guard", g(fr, "overlap_iou_guard", default=None)),
        ("overlap_guard_pre", g(fr, "overlap_guard_pre", default=None)),
        ("overlap_guard", g(fr, "overlap_guard", default=None)),
        ("overlap_freeze", g(fr, "overlap_freeze", default=None)),
        ("overlap_freeze_reason", g(fr, "overlap_freeze_reason", default=None)),
        ("overlap_freeze_side", g(fr, "overlap_freeze_side", default=None)),
        ("overlap_z_hint", g(fr, "overlap_z_hint", default=None)),
        ("overlap_z_abs", g(fr, "overlap_z_abs", default=None)),
        ("overlap_hand_z_diff", g(fr, "overlap_hand_z_diff", default=None)),
        ("occlusion_freeze_age_1", g(fr, "occlusion_freeze_age_1", default=None)),
        ("occlusion_freeze_age_2", g(fr, "occlusion_freeze_age_2", default=None)),
        ("occlusion_freeze_max", g(fr, "occlusion_freeze_max", default=None)),
        ("pose_amb", g(fr, "pose_side_ambiguous", default=None)),
        ("pose_side_ratio", g(fr, "pose_side_ratio", default=None)),
        ("side_d_ll", g(fr, "side_d_ll", default=None)),
        ("side_d_lr", g(fr, "side_d_lr", default=None)),
        ("side_d_rl", g(fr, "side_d_rl", default=None)),
        ("side_d_rr", g(fr, "side_d_rr", default=None)),
        ("side_cost_current", g(fr, "side_cost_current", default=None)),
        ("side_cost_swap", g(fr, "side_cost_swap", default=None)),
        ("side_pref_left", g(fr, "side_pref_left", default=None)),
        ("side_pref_right", g(fr, "side_pref_right", default=None)),
        ("pose_interpolated", g(fr, "pose_interpolated", default=None)),
    ]
    rows = []
    for key, val in items:
        rows.append({"metric": key, "value": _fmt(val, nd=4)})
    return pd.DataFrame(rows)


def _df_html(df):
    if df is None:
        return ""
    try:
        return df.to_html(index=False)
    except Exception:
        return ""
def render_compare(frame_idx, **kwargs):
    raw = render_frame(frame_idx, run="raw", **kwargs)
    pp = render_frame(frame_idx, run="pp", **kwargs)
    if raw is None:
        return None
    if pp is None:
        return raw
    h = max(raw.shape[0], pp.shape[0])
    w = raw.shape[1] + pp.shape[1]
    canvas = np.zeros((h, w, 3), dtype=np.uint8)
    canvas[:raw.shape[0], :raw.shape[1]] = raw
    canvas[:pp.shape[0], raw.shape[1]:raw.shape[1]+pp.shape[1]] = pp
    return canvas


def sp_candidates_table(fr, idx, top_n=10):
    h = get_hand(fr, idx)
    dbg = h.get("sp_debug") or {}
    cand = dbg.get("candidates") or []
    if not cand:
        return None
    df = pd.DataFrame(cand)
    if "combined" in df.columns:
        df = df.sort_values("combined", ascending=False)
    elif "score" in df.columns:
        df = df.sort_values("score", ascending=False)
    return df.head(top_n)


## E) Interactive UI (single / grid / compare)

In [5]:
if not HAS_WIDGETS:
    print("ipywidgets not available; use render_frame(idx) manually.")
else:
    # Reset previous UI instances
    if "_ui_widgets" in globals():
        for w in _ui_widgets:
            try:
                w.close()
            except Exception:
                pass
    _ui_widgets = []
    clear_output(wait=True)
    mode_widget = widgets.ToggleButtons(options=["single", "grid", "compare"], value="single", description="mode")
    run_widget = widgets.Dropdown(options=["raw"], value="raw", description="run")
    analytics_run = widgets.Dropdown(options=["raw"], value="raw", description="analytics")
    idx_slider = widgets.IntSlider(min=0, max=max(0, len(RUNS["raw"]["frames"]) - 1), value=0, description="idx", readout=False, continuous_update=False)
    idx_label = widgets.HTML(value="")
    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next")
    show_pose = widgets.Checkbox(value=True, description="pose")
    show_hands = widgets.Checkbox(value=True, description="hands")
    show_sp = widgets.Checkbox(value=True, description="second pass")
    show_text = widgets.Checkbox(value=True, description="text")
    verbose = widgets.Checkbox(value=False, description="verbose")
    zoom_widget = widgets.FloatSlider(value=1.0, min=0.25, max=2.5, step=0.05, description="zoom")

    grid_n = widgets.BoundedIntText(value=12, min=1, max=200, description="n")
    grid_cols = widgets.BoundedIntText(value=4, min=1, max=12, description="cols")
    grid_scale = widgets.FloatSlider(value=0.6, min=0.2, max=1.0, step=0.05, description="scale")
    grid_mode = widgets.Dropdown(options=["even", "around_events"], value="even", description="grid_mode")
    grid_event = widgets.Dropdown(options=["sanity", "swap", "dedup", "sp_recovered", "sp_attempt", "occlusion_saved", "missing_gap", "outlier", "overlap_guard", "overlap_freeze", "pose_side_drop", "pose_side_reassign", "occ_freeze", "pp_applied", "pp_diff"], description="event")
    grid_window = widgets.BoundedIntText(value=2, min=0, max=50, description="event_window")


    status_html = widgets.HTML(value="")


    status_html = widgets.HTML(value="")
    legend_html = widgets.HTML(value=""")
    <b>Legend</b>:
    <ul>
      <li><b>source</b>: pass1/pass2=detected, tracked=tracker, occluded=filled from last good, hold=interp-hold, interp=postprocess</li>
      <li><b>state</b>: observed=detected, predicted=tracked/interp, occluded=filled, missing=none</li>
      <li><b>anchor</b>: score &ge; HI gate; used for tracker init/postprocess anchors</li>
      <li><b>occl_ttl</b>: occlusion TTL counter; track_age = frames since last detection</li>
      <li><b>ROI</b>/<b>hint</b>: second-pass search region/center; SEL=selected ROI</li>
<li><b>pose_q</b>: wrist-to-pose quality (1/(1+dist_norm)); higher is better</li>
    </ul>
    """)

    out_img = widgets.Output()
    out_table = widgets.Output()
    frame_html = widgets.HTML(value="")

    _sync_guard = {"busy": False}

    def _sync_compare_from_checkbox(change):
        if _sync_guard["busy"]:
            return
        _sync_guard["busy"] = True
        try:
            if change["new"]:
                mode_widget.value = "compare"
            elif mode_widget.value == "compare":
                mode_widget.value = "single"
        finally:
            _sync_guard["busy"] = False

    def _sync_compare_from_mode(change):
        if _sync_guard["busy"]:
            return
        _sync_guard["busy"] = True
        try:
            if compare_widget:
                compare_widget.value = (mode_widget.value == "compare")
        finally:
            _sync_guard["busy"] = False

    if compare_widget:
        compare_widget.observe(_sync_compare_from_checkbox, "value")
    mode_widget.observe(_sync_compare_from_mode, "value")


    def _safe_idx(val, fallback=0):
        try:
            if val is None:
                return int(fallback)
            if isinstance(val, float) and (val != val):
                return int(fallback)
            return int(val)
        except Exception:
            return int(fallback)


    PP_DIFF = []

    def _safe_int(val, fallback=0):
        try:
            if val is None:
                return int(fallback)
            if isinstance(val, float) and (val != val):
                return int(fallback)
            return int(val)
        except Exception:
            return int(fallback)

    def _safe_float(val, fallback=0.0):
        try:
            if val is None:
                return float(fallback)
            if isinstance(val, float) and (val != val):
                return float(fallback)
            return float(val)
        except Exception:
            return float(fallback)


    _event_sync = {"busy": False}

    def _sync_event_to_timeline(_=None):
        if _event_sync["busy"]:
            return
        ev = None
        try:
            ev = grid_event.value
        except Exception:
            ev = None
        if ev is None:
            return
        if "ev_dropdown" not in globals():
            return
        if ev_dropdown is None:
            return
        if ev in ev_dropdown.options:
            _event_sync["busy"] = True
            try:
                ev_dropdown.value = ev
            finally:
                _event_sync["busy"] = False

    def _update_cache_reset(*_):
        global RUNS, VIDEO_CACHE
        RUNS = load_all_runs()
        meta = RUNS["raw"]["meta"]
        video_path = None
        if video_path_widget and video_path_widget.value:
            video_path = Path(video_path_widget.value)
        if not video_path:
            video_path = guess_video_path(meta, RUNS["raw"]["path"])
        VIDEO_CACHE = VideoCache(video_path) if video_path else None

        options = ["raw"] + (["pp"] if RUNS.get("pp") is not None else [])
        run_widget.options = options
        if run_widget.value not in options:
            run_widget.value = options[0]

        analytics_options = list(options)
        if len(options) > 1:
            analytics_options.append("both")
        analytics_run.options = analytics_options
        if analytics_run.value not in analytics_options:
            analytics_run.value = analytics_options[0]

        max_idx = len(RUNS["raw"]["frames"]) - 1
        if RUNS.get("pp") is not None:
            max_idx = min(max_idx, len(RUNS["pp"]["frames"]) - 1)
        idx_slider.max = max(0, max_idx)
        idx_slider.value = min(_safe_idx(idx_slider.value, 0), idx_slider.max)
        grid_n.max = max(1, idx_slider.max + 1)
        grid_n.value = max(1, min(_safe_int(grid_n.value, 12), grid_n.max))
        grid_cols.value = max(1, min(_safe_int(grid_cols.value, 4), 12))
        RENDER_CACHE.clear()

        # status summary for raw/pp
        if RUNS.get("pp") is not None:
            raw_frames = RUNS["raw"]["frames"]
            pp_frames = RUNS["pp"]["frames"]
            stats = pp_diff_stats(raw_frames, pp_frames)
            PP_DIFF.clear()
            PP_DIFF.extend(pp_diff_indices(raw_frames, pp_frames))
            status_html.value = (
                f"<b>raw</b>: {len(raw_frames)} frames | <b>pp</b>: {len(pp_frames)} frames "
                f"| diff_frames: {stats['diff_frames']} | pp_applied: {stats['pp_applied']} "
                f"| pp_overrode: {stats['pp_overrode']}"
            )
        else:
            PP_DIFF.clear()
            status_html.value = "pp not loaded"

        refresh = globals().get("refresh_timeline")
        if callable(refresh):
            refresh()
        analytics_refresh = globals().get("_analytics_refresh_impl")
        if callable(analytics_refresh):
            analytics_refresh()
        _sync_event_to_timeline()
        _render()

    if json_path_widget:
        json_path_widget.observe(_update_cache_reset, "value")
    if analytics_run:
        analytics_run.observe(lambda _: globals().get("refresh_timeline", lambda: None)(), "value")
    if analytics_run:
        analytics_run.observe(lambda _: globals().get("_analytics_refresh_impl", lambda: None)(), "value")
    if use_pp_widget:
        use_pp_widget.observe(_update_cache_reset, "value")
    if compare_widget:
        compare_widget.observe(_update_cache_reset, "value")
    if video_path_widget:
        video_path_widget.observe(_update_cache_reset, "value")
    if tune_select_widget:
        tune_select_widget.observe(_update_cache_reset, "value")

    def _render(*_):
        out_img.clear_output(wait=True)
        with out_img:
            mode = mode_widget.value
            if compare_widget and compare_widget.value:
                mode = "compare"

            safe_idx = _safe_idx(idx_slider.value, 0)
            idx_label.value = f"<b>idx</b>: {safe_idx}/{idx_slider.max}"
            if mode == "compare" and RUNS.get("pp") is not None:
                img = render_compare(safe_idx, show_pose=show_pose.value, show_hands=show_hands.value,
                                     show_sp=show_sp.value, show_text=show_text.value, verbose=verbose.value)
            elif mode == "grid":
                frames = RUNS[run_widget.value]["frames"]
                idxs = []
                n_val = max(1, min(_safe_int(grid_n.value, 12), len(frames)))
                win = max(0, _safe_int(grid_window.value, 2))
                if grid_mode.value == "even":
                    if n_val > 0:
                        idxs = np.linspace(0, len(frames) - 1, n_val).astype(int).tolist()
                else:
                    build_events_fn = globals().get("build_events")
                    build_timeseries_fn = globals().get("build_timeseries")
                    if callable(build_events_fn) and callable(build_timeseries_fn):
                        ev = build_events_fn(build_timeseries_fn(frames))
                    else:
                        ev = {}
                    if grid_event.value == "pp_diff" and PP_DIFF:
                        hits = PP_DIFF
                    else:
                        hits = ev.get(grid_event.value, [])
                    if not hits:
                        # fallback to even grid if no events
                        idxs = np.linspace(0, len(frames) - 1, n_val).astype(int).tolist()
                    else:
                        for h in hits:
                            idxs.extend(list(range(max(0, h - win), min(len(frames), h + win + 1))))
                        idxs = sorted(set(idxs))[: n_val]
                if not idxs:
                    idxs = [_safe_idx(idx_slider.value, 0)]
                imgs = []
                for i in idxs:
                    im = render_frame(i, run=run_widget.value, show_pose=show_pose.value, show_hands=show_hands.value,
                                      show_sp=show_sp.value, show_text=show_text.value, verbose=verbose.value)
                    if im is not None:
                        # label each tile with index
                        cv2.putText(im, f"#{i}", (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1, cv2.LINE_AA)
                        imgs.append(im)
                imgs = [im for im in imgs if im is not None]
                if not imgs:
                    return
                cols = max(1, min(_safe_int(grid_cols.value, 4), 12))
                rows = int(math.ceil(len(imgs) / cols))
                h = max(im.shape[0] for im in imgs)
                w = max(im.shape[1] for im in imgs)
                canvas = np.zeros((rows * h, cols * w, 3), dtype=np.uint8)
                for i, im in enumerate(imgs):
                    r = i // cols
                    c = i % cols
                    canvas[r*h:r*h+im.shape[0], c*w:c*w+im.shape[1]] = im
                scale = _safe_float(grid_scale.value, 0.6)
                max_w = 1600
                if canvas.shape[1] > max_w:
                    scale = min(scale, max_w / float(canvas.shape[1]))
                if scale < 0.99:
                    new_w = max(1, int(canvas.shape[1] * scale))
                    new_h = max(1, int(canvas.shape[0] * scale))
                    canvas = cv2.resize(canvas, (new_w, new_h), interpolation=cv2.INTER_AREA)
                img = canvas
            else:
                img = render_frame(safe_idx, run=run_widget.value, show_pose=show_pose.value, show_hands=show_hands.value,
                                   show_sp=show_sp.value, show_text=show_text.value, verbose=verbose.value)
            if img is not None:
                show_image(img, zoom=_safe_float(zoom_widget.value, 1.0))

        out_table.clear_output(wait=True)
        with out_table:
            idx_label.value = f"<b>idx</b>: {safe_idx}/{idx_slider.max}"
            if mode == "compare" and RUNS.get("pp") is not None:
                fr_raw = RUNS["raw"]["frames"][safe_idx]
                fr_pp = RUNS["pp"]["frames"][safe_idx]
                diff_df = compare_diff(fr_raw, fr_pp)
                if diff_df is not None:
                    print("Diff (raw vs pp)")
                    display(diff_df)
                for label, fr in (("raw", fr_raw), ("pp", fr_pp)):
                    for idx in (1, 2):
                        df = sp_candidates_table(fr, idx)
                        if df is not None:
                            print(f"SP candidates {label} H{idx}")
                            display(df)
            elif mode != "grid":
                fr = RUNS[run_widget.value]["frames"][safe_idx]
                for idx in (1, 2):
                    df = sp_candidates_table(fr, idx)
                    if df is not None:
                        print(f"SP candidates {run_widget.value} H{idx}")
                        display(df)

        if mode != "grid":
            if mode == "compare" and RUNS.get("pp") is not None:
                fr_raw = RUNS["raw"]["frames"][safe_idx]
                fr_pp = RUNS["pp"]["frames"][safe_idx]
                table_raw, header_raw = frame_summary_table(fr_raw, idx=safe_idx)
                table_pp, header_pp = frame_summary_table(fr_pp, idx=safe_idx)
                cfg_raw = frame_config_table(fr_raw)
                cfg_pp = frame_config_table(fr_pp)
                decision_raw = decision_trace_table(fr_raw)
                decision_pp = decision_trace_table(fr_pp)
                dbg_raw = frame_debug_table(fr_raw) if verbose.value else None
                dbg_pp = frame_debug_table(fr_pp) if verbose.value else None
                frame_html.value = (
                    f"<pre>Frame header (raw) {header_raw}</pre>" + _df_html(table_raw) +
                    "<pre>Config (raw)</pre>" + _df_html(cfg_raw) +
                    "<pre>Decision trace (raw)</pre>" + _df_html(decision_raw) +
                    ("<pre>Frame debug (raw)</pre>" + _df_html(dbg_raw) if verbose.value else "") +
                    f"<pre>Frame header (pp) {header_pp}</pre>" + _df_html(table_pp) +
                    "<pre>Config (pp)</pre>" + _df_html(cfg_pp) +
                    "<pre>Decision trace (pp)</pre>" + _df_html(decision_pp) +
                    ("<pre>Frame debug (pp)</pre>" + _df_html(dbg_pp) if verbose.value else "")
                )
            else:
                fr = RUNS[run_widget.value]["frames"][safe_idx]
                table, header = frame_summary_table(fr, idx=safe_idx)
                cfg = frame_config_table(fr)
                decision = decision_trace_table(fr)
                dbg = frame_debug_table(fr) if verbose.value else None
                frame_html.value = (
                    f"<pre>Frame header {header}</pre>" + _df_html(table) +
                    "<pre>Config</pre>" + _df_html(cfg) +
                    "<pre>Decision trace</pre>" + _df_html(decision) +
                    ("<pre>Frame debug</pre>" + _df_html(dbg) if verbose.value else "")
                )
        else:
            frame_html.value = ""

    def _prev(_):
        idx_slider.value = max(0, _safe_idx(idx_slider.value, 0) - 1)

    def _next(_):
        idx_slider.value = min(idx_slider.max, _safe_idx(idx_slider.value, 0) + 1)

    prev_btn.on_click(_prev)
    next_btn.on_click(_next)

    def _install_keybinds():
        js = """
        (function() {
          if (window.__kp_keybinds_installed__) { return; }
          window.__kp_keybinds_installed__ = true;
          document.addEventListener('keydown', function(ev) {
            if (ev.defaultPrevented) { return; }
            var key = ev.key;
            if (key !== 'ArrowLeft' && key !== 'ArrowRight') { return; }
            var tag = (ev.target && ev.target.tagName) ? ev.target.tagName.toLowerCase() : '';
            if (tag === 'input' || tag === 'textarea' || tag === 'select') { return; }
            if (window.Jupyter && Jupyter.notebook && Jupyter.notebook.kernel) {
              if (key === 'ArrowLeft') {
                Jupyter.notebook.kernel.execute('_prev(None)');
              } else {
                Jupyter.notebook.kernel.execute('_next(None)');
              }
              ev.preventDefault();
            }
          }, true);
        })();
        """
        display(Javascript(js))

    _install_keybinds()

    def _coerce_idx(change):
        new = change.get("new")
        if new is None:
            idx_slider.value = 0
            return
        if isinstance(new, float) and math.isnan(new):
            idx_slider.value = 0
            return

    idx_slider.observe(_coerce_idx, "value")

    for w in [mode_widget, run_widget, analytics_run, idx_slider, show_pose, show_hands, show_sp, show_text, verbose,
              grid_n, grid_cols, grid_mode, grid_event, grid_window]:
        w.observe(_render, "value")

    _ui_box = widgets.VBox([
        widgets.HBox([mode_widget, run_widget, analytics_run, idx_slider, idx_label, prev_btn, next_btn]),
        widgets.HBox([show_pose, show_hands, show_sp, show_text, verbose, zoom_widget]),
        widgets.HBox([grid_n, grid_cols, grid_scale, grid_mode, grid_event, grid_window]),
        status_html,
        legend_html,
        out_img,
        out_table,
        frame_html,
    ])

    _ui_widgets = [mode_widget, run_widget, analytics_run, idx_slider, idx_label, prev_btn, next_btn,
                  show_pose, show_hands, show_sp, show_text, verbose, zoom_widget,
                  grid_n, grid_cols, grid_scale, grid_mode, grid_event, grid_window,
                  status_html, legend_html, out_img, out_table, frame_html, _ui_box]

    display(_ui_box)

    _update_cache_reset()

<IPython.core.display.Javascript object>

IndexError: list index out of range

## F) Diagnostics timeline & jump-to-events

In [ ]:
def _mean_l2_xy(a, b):
    if not a or not b:
        return None
    L = min(len(a), len(b))
    if L <= 0:
        return None
    s = 0.0
    for j in range(L):
        ax, ay = float(a[j]["x"]), float(a[j]["y"])
        bx, by = float(b[j]["x"]), float(b[j]["y"])
        s += math.hypot(ax - bx, ay - by)
    return s / float(L)


def _hand_scale(pts):
    if not pts:
        return 0.0
    xs = [p["x"] for p in pts]
    ys = [p["y"] for p in pts]
    if not xs or not ys:
        return 0.0
    return max(max(xs) - min(xs), max(ys) - min(ys))


def _outlier_proxy(cur, prev):
    if not cur or not prev:
        return False
    dist = _mean_l2_xy(cur, prev)
    if dist is None:
        return False
    prev_scale = _hand_scale(prev)
    cur_scale = _hand_scale(cur)
    denom = max(prev_scale, 1e-6)
    jump = dist / denom
    scale_ratio = None
    if prev_scale > 0.0 and cur_scale > 0.0:
        scale_ratio = max(prev_scale, cur_scale) / min(prev_scale, cur_scale)
    return jump > 2.5 or (scale_ratio is not None and scale_ratio > 1.6)


def build_timeseries(frames):
    rows = []
    prev1 = None
    prev2 = None
    for i, fr in enumerate(frames):
        h1 = get_hand(fr, 1)
        h2 = get_hand(fr, 2)
        outlier1 = _outlier_proxy(h1["pts"], prev1)
        outlier2 = _outlier_proxy(h2["pts"], prev2)
        prev1 = h1["pts"] if h1["pts"] is not None else None
        prev2 = h2["pts"] if h2["pts"] is not None else None

        missing1 = (h1["pts"] is None) or (h1["state"] in (None, "missing"))
        missing2 = (h2["pts"] is None) or (h2["state"] in (None, "missing"))

        reason1 = h1.get("reject")
        reason2 = h2.get("reject")
        pose_side_drop = _has_reason(reason1, "pose_side_drop") or _has_reason(reason2, "pose_side_drop")
        pose_side_reassign = _has_reason(reason1, "pose_side_reassign") or _has_reason(reason2, "pose_side_reassign")
        occ_freeze = _has_reason(reason1, "occ_freeze") or _has_reason(reason2, "occ_freeze")

        rows.append({
            "idx": i,
            "ts": g(fr, "ts", default=None),
            "score1": h1["score"],
            "score2": h2["score"],
            "score_gate1": h1.get("score_gate"),
            "score_gate2": h2.get("score_gate"),
            "pose_q1": h1.get("pose_quality"),
            "pose_q2": h2.get("pose_quality"),
            "pose_wrist_dist1": g(fr, "pose_wrist_dist_1", default=None),
            "pose_wrist_dist2": g(fr, "pose_wrist_dist_2", default=None),
            "state1": h1["state"],
            "state2": h2["state"],
            "source1": h1["source"],
            "source2": h2["source"],
            "occluded_ttl1": h1["occluded_ttl"],
            "occluded_ttl2": h2["occluded_ttl"],
            "track_age1": h1["track_age"],
            "track_age2": h2["track_age"],
            "track_reset1": bool(h1.get("track_reset")),
            "track_reset2": bool(h2.get("track_reset")),
            "tracker_last_score1": h1.get("tracker_last_score"),
            "tracker_last_score2": h2.get("tracker_last_score"),
            "overlap_iou": g(fr, "sp_overlap_iou", "overlap_iou", default=None),
            "overlap_iou_guard": g(fr, "overlap_iou_guard", default=None),
            "overlap_guard": bool(g(fr, "overlap_guard", default=False)),
            "overlap_freeze": bool(g(fr, "overlap_freeze", default=False)),
            "overlap_freeze_side": g(fr, "overlap_freeze_side", default=None),
            "overlap_z_abs": g(fr, "overlap_z_abs", default=None),
            "overlap_z_hint": g(fr, "overlap_z_hint", default=None),
            "overlap_hand_z_diff": g(fr, "overlap_hand_z_diff", default=None),
            "occl_freeze_age1": h1.get("occlusion_freeze_age"),
            "occl_freeze_age2": h2.get("occlusion_freeze_age"),
            "occl_freeze_max": h1.get("occlusion_freeze_max") or g(fr, "occlusion_freeze_max", default=None),
            "pose_amb": bool(g(fr, "pose_side_ambiguous", default=False)),
            "pose_side_ratio": g(fr, "pose_side_ratio", default=None),
            "side_ok1": h1.get("side_ok"),
            "side_ok2": h2.get("side_ok"),
            "side_cost_current": g(fr, "side_cost_current", default=None),
            "side_cost_swap": g(fr, "side_cost_swap", default=None),
            "sanity_bone_err1": h1.get("sanity_bone_max_err"),
            "sanity_bone_err2": h2.get("sanity_bone_max_err"),
            "sanity_scale_ratio1": h1.get("sanity_scale_ratio"),
            "sanity_scale_ratio2": h2.get("sanity_scale_ratio"),
            "swap": bool(g(fr, "swap_applied", default=False)),
            "dedup": bool(g(fr, "dedup_triggered", default=False)),
            "sp_attempt": bool(h1["sp_attempt"]) or bool(h2["sp_attempt"]),
            "sp_recovered": bool(h1["sp_recovered"]) or bool(h2["sp_recovered"]),
            "sanity_reject": is_sanity_reject(h1["reject"]) or is_sanity_reject(h2["reject"]),
            "pose_side_drop": bool(pose_side_drop),
            "pose_side_reassign": bool(pose_side_reassign),
            "occ_freeze": bool(occ_freeze),
            "pp_applied": bool(h1["pp_applied"]) or bool(h2["pp_applied"]),
            "occlusion_saved": bool(h1["occlusion_saved"]) or bool(h2["occlusion_saved"]),
            "outlier_proxy": bool(outlier1) or bool(outlier2),
            "missing_any": bool(missing1 or missing2),
        })
    return pd.DataFrame(rows)

def plot_timeline(df, hand="both", highlight=None, highlight_label=None, pose_qual_min=None):
    if df is None or df.empty:
        print("No data")
        return
    fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
    if hand in ("both", 1):
        axes[0].plot(df["idx"], df["score1"], label="score1")
    if hand in ("both", 2):
        axes[0].plot(df["idx"], df["score2"], label="score2")
    if "pose_q1" in df.columns and hand in ("both", 1):
        axes[0].plot(df["idx"], df["pose_q1"], label="pose_q1", linestyle="--", alpha=0.6)
    if "pose_q2" in df.columns and hand in ("both", 2):
        axes[0].plot(df["idx"], df["pose_q2"], label="pose_q2", linestyle="--", alpha=0.6)
    if pose_qual_min is not None:
        axes[0].axhline(float(pose_qual_min), color="gray", linestyle=":", alpha=0.4, label="pose_q_min")
    axes[0].legend(loc="upper left", bbox_to_anchor=(1.05, 1.0), borderaxespad=0.0)
    axes[0].set_ylabel("score / pose_q")

    if hand in ("both", 1):
        axes[1].plot(df["idx"], df["occluded_ttl1"], label="occl_ttl1")
        axes[1].plot(df["idx"], df["track_age1"], label="track_age1")
        if "occl_freeze_age1" in df.columns:
            axes[1].plot(df["idx"], df["occl_freeze_age1"], label="occ_freeze_age1", linestyle="--", alpha=0.6)
    if hand in ("both", 2):
        axes[1].plot(df["idx"], df["occluded_ttl2"], label="occl_ttl2")
        axes[1].plot(df["idx"], df["track_age2"], label="track_age2")
        if "occl_freeze_age2" in df.columns:
            axes[1].plot(df["idx"], df["occl_freeze_age2"], label="occ_freeze_age2", linestyle=":", alpha=0.6)

    # overlap iou
    axes[1].plot(df["idx"], df["overlap_iou"], label="overlap_iou", alpha=0.5)
    if "overlap_iou_guard" in df.columns:
        axes[1].plot(df["idx"], df["overlap_iou_guard"], label="overlap_iou_guard", alpha=0.4, linestyle=":")

    # markers
    axes[1].scatter(df.loc[df["swap"], "idx"], [0]*df["swap"].sum(), marker="x", color="red", label="swap")
    axes[1].scatter(df.loc[df["dedup"], "idx"], [0.1]*df["dedup"].sum(), marker="x", color="purple", label="dedup")
    axes[1].scatter(df.loc[df["sp_recovered"], "idx"], [0.2]*df["sp_recovered"].sum(), marker="o", color="green", label="sp_rec")
    axes[1].scatter(df.loc[df["sp_attempt"], "idx"], [0.25]*df["sp_attempt"].sum(), marker=".", color="green", label="sp_att")
    axes[1].scatter(df.loc[df["sanity_reject"], "idx"], [0.3]*df["sanity_reject"].sum(), marker="^", color="orange", label="sanity")
    axes[1].scatter(df.loc[df["pp_applied"], "idx"], [0.35]*df["pp_applied"].sum(), marker="s", color="blue", label="pp")
    if "overlap_guard" in df.columns:
        axes[1].scatter(df.loc[df["overlap_guard"], "idx"], [0.4]*df["overlap_guard"].sum(), marker="|", color="black", label="overlap_guard")
    if "overlap_freeze" in df.columns:
        axes[1].scatter(df.loc[df["overlap_freeze"], "idx"], [0.45]*df["overlap_freeze"].sum(), marker="|", color="brown", label="overlap_freeze")
    if "pose_side_drop" in df.columns:
        axes[1].scatter(df.loc[df["pose_side_drop"], "idx"], [0.5]*df["pose_side_drop"].sum(), marker="|", color="gray", label="pose_side_drop")
    if "pose_side_reassign" in df.columns:
        axes[1].scatter(df.loc[df["pose_side_reassign"], "idx"], [0.55]*df["pose_side_reassign"].sum(), marker="|", color="magenta", label="pose_side_reassign")
    if "occ_freeze" in df.columns:
        axes[1].scatter(df.loc[df["occ_freeze"], "idx"], [0.6]*df["occ_freeze"].sum(), marker="|", color="brown", label="occ_freeze")

    # highlight selected event
    if highlight:
        xs = highlight
        if len(xs) > 300:
            step = max(1, len(xs) // 300)
            xs = xs[::step]
        y0 = axes[0].get_ylim()[1]
        axes[0].scatter(xs, [y0] * len(xs), marker="|", color="black", label=highlight_label or "event")
        y1 = axes[1].get_ylim()[1]
        axes[1].scatter(xs, [y1] * len(xs), marker="|", color="black")

    axes[1].legend(loc="upper left", bbox_to_anchor=(1.05, 1.0), borderaxespad=0.0)
    axes[1].set_ylabel("ttl / track / overlap")
    axes[1].set_xlabel("frame")
    fig.subplots_adjust(right=0.72)
    plt.show()

def build_events(df: pd.DataFrame):
    events = {}
    if df is None or df.empty:
        return events
    events["sanity"] = df.loc[df["sanity_reject"], "idx"].tolist()
    events["swap"] = df.loc[df["swap"], "idx"].tolist()
    events["dedup"] = df.loc[df["dedup"], "idx"].tolist()
    events["sp_recovered"] = df.loc[df["sp_recovered"], "idx"].tolist()
    events["sp_attempt"] = df.loc[df["sp_attempt"], "idx"].tolist()
    events["occlusion_saved"] = df.loc[df["occlusion_saved"], "idx"].tolist()
    events["outlier"] = df.loc[df["outlier_proxy"], "idx"].tolist()
    events["overlap_guard"] = df.loc[df["overlap_guard"], "idx"].tolist() if "overlap_guard" in df.columns else []
    events["overlap_freeze"] = df.loc[df["overlap_freeze"], "idx"].tolist() if "overlap_freeze" in df.columns else []
    events["pose_side_drop"] = df.loc[df["pose_side_drop"], "idx"].tolist() if "pose_side_drop" in df.columns else []
    events["pose_side_reassign"] = df.loc[df["pose_side_reassign"], "idx"].tolist() if "pose_side_reassign" in df.columns else []
    events["occ_freeze"] = df.loc[df["occ_freeze"], "idx"].tolist() if "occ_freeze" in df.columns else []
    if "pp_applied" in df.columns:
        events["pp_applied"] = df.loc[df["pp_applied"], "idx"].tolist()

    # missing gap >=5
    gaps = []
    cur = []
    for i, m in enumerate(df["missing_any"].tolist()):
        if m:
            cur.append(i)
        else:
            if len(cur) >= 5:
                gaps.extend(cur)
            cur = []
    if len(cur) >= 5:
        gaps.extend(cur)
    events["missing_gap"] = gaps
    return events

def _analytics_run_name():
    run_name = "raw"
    if "analytics_run" in globals() and analytics_run is not None:
        try:
            run_name = analytics_run.value
        except Exception:
            run_name = "raw"
    if run_name == "both":
        return "both"
    if RUNS.get(run_name) is None:
        run_name = "raw"
    return run_name


def _timeline_frames():
    run_name = _analytics_run_name()
    if run_name == "both":
        frames_raw = RUNS.get("raw", {}).get("frames", [])
        frames_pp = RUNS.get("pp", {}).get("frames", [])
        return frames_raw, frames_pp
    return RUNS[run_name]["frames"]


def refresh_timeline():
    global df, df_pp, event_map, event_map_pp, timeline_run_name
    timeline_run_name = _analytics_run_name()
    frames = _timeline_frames()
    df_pp = None
    event_map_pp = {}
    if isinstance(frames, tuple):
        frames_raw, frames_pp = frames
        df = build_timeseries(frames_raw)
        event_map = build_events(df)
        if frames_pp:
            df_pp = build_timeseries(frames_pp)
            event_map_pp = build_events(df_pp)
    else:
        df = build_timeseries(frames)
        event_map = build_events(df)
    if HAS_WIDGETS and "ev_dropdown" in globals():
        keys = set(event_map.keys())
        if event_map_pp:
            keys |= set(event_map_pp.keys())
        ev_dropdown.options = list(keys)
        _plot()


if HAS_WIDGETS:
    # Reset previous timeline UI
    if "_timeline_widgets" in globals():
        for w in _timeline_widgets:
            try:
                w.close()
            except Exception:
                pass
    clear_output(wait=True)

    df = None
    df_pp = None
    event_map = {}
    event_map_pp = {}
    refresh_timeline()

    ev_dropdown = widgets.Dropdown(options=list(event_map.keys()), description="event")
    ev_prev = widgets.Button(description="Prev")
    ev_next = widgets.Button(description="Next")
    ev_label = widgets.Label(value="0/0")
    timeline_out = widgets.Output()

    def _plot(*_):
        with timeline_out:
            clear_output(wait=True)
            ev = ev_dropdown.value if ev_dropdown else None
            hits = event_map.get(ev, []) if ev else []
            hits_pp = event_map_pp.get(ev, []) if ev else []
            run_name = globals().get("timeline_run_name", "raw")
            if run_name != "both":
                meta = RUNS.get(run_name, {}).get("meta") or {}
                pose_min = None
                if isinstance(meta, dict):
                    pose_min = (meta.get("hand_score_gate") or {}).get("pose_dist_qual_min")
                if df is not None:
                    print(run_name)
                    plot_timeline(df, highlight=hits, highlight_label=ev, pose_qual_min=pose_min)
                if ev_label is not None:
                    ev_label.value = f"{len(hits)} events"
            else:
                meta_raw = RUNS.get("raw", {}).get("meta") or {}
                meta_pp = RUNS.get("pp", {}).get("meta") or {}
                pose_min_raw = None
                pose_min_pp = None
                if isinstance(meta_raw, dict):
                    pose_min_raw = (meta_raw.get("hand_score_gate") or {}).get("pose_dist_qual_min")
                if isinstance(meta_pp, dict):
                    pose_min_pp = (meta_pp.get("hand_score_gate") or {}).get("pose_dist_qual_min")
                if df is not None:
                    print("raw")
                    plot_timeline(df, highlight=hits, highlight_label=ev, pose_qual_min=pose_min_raw)
                if df_pp is not None:
                    print("pp")
                    plot_timeline(df_pp, highlight=hits_pp, highlight_label=ev, pose_qual_min=pose_min_pp)
                if ev_label is not None:
                    ev_label.value = f"raw {len(hits)} | pp {len(hits_pp)}"

    _plot()

    state = {"idx": 0}

    def _jump(delta):
        ev = ev_dropdown.value
        hits = event_map.get(ev, [])
        if not hits:
            ev_label.value = "0/0 (0 events)"
            return
        state["idx"] = max(0, min(len(hits) - 1, state["idx"] + delta))
        idx = hits[state["idx"]]
        if idx is None:
            return
        ev_label.value = f"{state['idx']+1}/{len(hits)} -> {idx}"
        try:
            idx_slider.value = int(idx)
        except Exception:
            return

    ev_prev.on_click(lambda _: _jump(-1))
    ev_next.on_click(lambda _: _jump(1))
    ev_dropdown.observe(_plot, "value")

    _timeline_box = widgets.VBox([
        widgets.HBox([ev_dropdown, ev_prev, ev_next, ev_label]),
        timeline_out,
    ])
    _timeline_widgets = [ev_dropdown, ev_prev, ev_next, ev_label, timeline_out, _timeline_box]
    if "_timeline_handle" in globals():
        try:
            _timeline_handle.update(_timeline_box)
        except Exception:
            _timeline_handle = display(_timeline_box, display_id=True)
    else:
        _timeline_handle = display(_timeline_box, display_id=True)


if HAS_WIDGETS and "run_widget" in globals() and not globals().get("_timeline_run_observer"):
    def _on_run_change(_):
        refresh_timeline()
    run_widget.observe(_on_run_change, "value")
    globals()["_timeline_run_observer"] = True


## G) Analytics summary


In [ ]:
from collections import Counter


def parse_sanity_reasons(reason: str | None):
    if not reason:
        return []
    s = str(reason)
    out = []
    for chunk in s.split(";"):
        if "sanity:" in chunk:
            tail = chunk.split("sanity:", 1)[1]
            for part in tail.split("|"):
                part = part.strip()
                if not part:
                    continue
                name = part.split()[0].strip()
                if name:
                    out.append(name)
    return out


def _state_from_source(state, source, pts_present):
    if state:
        return state
    if source in ("pass1", "pass2"):
        return "observed"
    if source in ("tracked", "interp", "hold"):
        return "predicted"
    if source == "occluded":
        return "occluded"
    if not pts_present:
        return "missing"
    return "missing"


def _pct(count, total):
    return round(100.0 * count / max(1, total), 2)


def compute_breakdowns(frames):
    frames_total = len(frames)
    if frames_total == 0:
        return {}

    sanity_counts = {1: Counter(), 2: Counter()}
    state_counts = {1: Counter(), 2: Counter()}
    source_counts = {1: Counter(), 2: Counter()}
    sp_attempt = {1: 0, 2: 0}
    sp_recovered = {1: 0, 2: 0}
    sp_recovered_pass2 = {1: 0, 2: 0}
    track_ok = {1: 0, 2: 0}
    track_ages = {1: [], 2: []}

    for fr in frames:
        for idx in (1, 2):
            h = get_hand(fr, idx)
            reason = h.get("reject")
            for r in parse_sanity_reasons(reason):
                sanity_counts[idx][r] += 1

            pts_present = h.get("pts") is not None
            st = _state_from_source(h.get("state"), h.get("source"), pts_present)
            src = h.get("source") or "none"
            state_counts[idx][st] += 1
            source_counts[idx][src] += 1

            if h.get("sp_attempt"):
                sp_attempt[idx] += 1
            if h.get("sp_recovered"):
                sp_recovered[idx] += 1
                if h.get("source") == "pass2":
                    sp_recovered_pass2[idx] += 1

            if h.get("track_ok"):
                track_ok[idx] += 1
                if h.get("track_age") is not None:
                    track_ages[idx].append(h.get("track_age"))

    return {
        "frames_total": frames_total,
        "sanity_counts": sanity_counts,
        "state_counts": state_counts,
        "source_counts": source_counts,
        "sp_attempt": sp_attempt,
        "sp_recovered": sp_recovered,
        "sp_recovered_pass2": sp_recovered_pass2,
        "track_ok": track_ok,
        "track_ages": track_ages,
    }


def table_from_counter(counter, total, label):
    rows = []
    for k, v in counter.most_common():
        rows.append({"item": k, "count": v, "pct": _pct(v, total), "hand": label})
    return pd.DataFrame(rows)


def event_samples(frames, max_n=10):
    df = build_timeseries(frames)
    ev = build_events(df)
    out = {}
    for name in ("sanity", "swap", "missing_gap", "sp_recovered"):
        hits = ev.get(name, [])
        if not hits:
            out[name] = []
            continue
        # take evenly spaced samples
        if len(hits) <= max_n:
            out[name] = hits
        else:
            step = max(1, len(hits) // max_n)
            out[name] = hits[::step][:max_n]
    return out


def render_analytics(frames):
    if not frames:
        print("No frames")
        return
    info = compute_breakdowns(frames)
    total = info["frames_total"]

    # Sanity breakdown
    sanity_rows = []
    for idx in (1, 2):
        df = table_from_counter(info["sanity_counts"][idx], total, f"hand{idx}")
        if not df.empty:
            sanity_rows.append(df)
    sanity_df = pd.concat(sanity_rows, ignore_index=True) if sanity_rows else pd.DataFrame()

    # State breakdown
    state_rows = []
    for idx in (1, 2):
        df = table_from_counter(info["state_counts"][idx], total, f"hand{idx}")
        if not df.empty:
            state_rows.append(df)
    state_df = pd.concat(state_rows, ignore_index=True) if state_rows else pd.DataFrame()

    # Source breakdown
    source_rows = []
    for idx in (1, 2):
        df = table_from_counter(info["source_counts"][idx], total, f"hand{idx}")
        if not df.empty:
            source_rows.append(df)
    source_df = pd.concat(source_rows, ignore_index=True) if source_rows else pd.DataFrame()

    # SP summary
    sp_rows = []
    for idx in (1, 2):
        att = info["sp_attempt"][idx]
        rec = info["sp_recovered"][idx]
        rec_p2 = info["sp_recovered_pass2"][idx]
        sp_rows.append({
            "hand": idx,
            "attempted": att,
            "recovered": rec,
            "recover_rate_%": _pct(rec, max(1, att)),
            "recovered_pass2": rec_p2,
        })
    sp_df = pd.DataFrame(sp_rows)

    # Tracking summary
    tr_rows = []
    for idx in (1, 2):
        ok = info["track_ok"][idx]
        ages = info["track_ages"][idx]
        mean_age = round(float(np.mean(ages)), 2) if ages else 0.0
        tr_rows.append({
            "hand": idx,
            "track_ok_frames": ok,
            "track_ok_%": _pct(ok, total),
            "track_age_mean": mean_age,
        })
    tr_df = pd.DataFrame(tr_rows)

    # Event samples
    samples = event_samples(frames)

    print(f"Frames: {total}")
    if not sanity_df.empty:
        print("\nSanity breakdown")
        display(sanity_df)
    else:
        print("\nSanity breakdown: none")

    print("\nState breakdown")
    display(state_df)

    print("\nSource breakdown")
    display(source_df)

    print("\nSecond-pass effectiveness")
    display(sp_df)

    print("\nTracking effectiveness")
    display(tr_df)

    print("\nEvent samples (frame indices)")
    display(pd.DataFrame({"event": list(samples.keys()), "frames": list(samples.values())}))

    # Simple bar charts
    if not sanity_df.empty:
        plt.figure(figsize=(8, 3))
        for hand in ("hand1", "hand2"):
            sub = sanity_df[sanity_df["hand"] == hand]
            if sub.empty:
                continue
            plt.bar([f"{hand}:{x}" for x in sub["item"]], sub["count"], alpha=0.6)
        plt.title("Sanity reasons counts")
        plt.xticks(rotation=30, ha="right")
        plt.show()

    if not state_df.empty:
        plt.figure(figsize=(8, 3))
        for hand in ("hand1", "hand2"):
            sub = state_df[state_df["hand"] == hand]
            plt.plot(sub["item"], sub["pct"], marker="o", label=hand)
        plt.title("State distribution (%)")
        plt.legend()
        plt.show()


if HAS_WIDGETS:
    # Reset previous analytics UI
    if "_analytics_widgets" in globals():
        for w in _analytics_widgets:
            try:
                w.close()
            except Exception:
                pass
    clear_output(wait=True)
    if "analytics_out" not in globals():
        analytics_out = widgets.Output()
    if "analytics_btn" not in globals():
        analytics_btn = widgets.Button(description="Recompute analytics")

    def _analytics_refresh_impl():
        with analytics_out:
            clear_output(wait=True)
            run_name = "raw"
            if "analytics_run" in globals() and analytics_run is not None:
                run_name = analytics_run.value
            if run_name == "both":
                frames_raw = RUNS.get("raw", {}).get("frames", [])
                frames_pp = RUNS.get("pp", {}).get("frames", [])
                print("RAW")
                render_analytics(frames_raw)
                if frames_pp:
                    print("PP")
                    render_analytics(frames_pp)
                else:
                    print("PP: not loaded")
            else:
                frames = RUNS.get(run_name, {}).get("frames", [])
                render_analytics(frames)

    globals()["_analytics_refresh_impl"] = _analytics_refresh_impl

    if not globals().get("_analytics_btn_bound"):
        def _analytics_on_click(_):
            fn = globals().get("_analytics_refresh_impl")
            if callable(fn):
                fn()
        analytics_btn.on_click(_analytics_on_click)
        globals()["_analytics_btn_bound"] = True

    if "run_widget" in globals() and not globals().get("_analytics_run_observer_raw"):
        def _on_run_change_raw(_):
            fn = globals().get("_analytics_refresh_impl")
            if callable(fn):
                fn()
        run_widget.observe(_on_run_change_raw, "value")
        globals()["_analytics_run_observer_raw"] = True

    if "analytics_run" in globals() and not globals().get("_analytics_run_observer"):
        def _on_run_change(_):
            fn = globals().get("_analytics_refresh_impl")
            if callable(fn):
                fn()
        analytics_run.observe(_on_run_change, "value")
        globals()["_analytics_run_observer"] = True

    _analytics_box = widgets.VBox([analytics_btn, analytics_out])
    _analytics_widgets = [analytics_btn, analytics_out, _analytics_box]
    if "_analytics_handle" in globals():
        try:
            _analytics_handle.update(_analytics_box)
        except Exception:
            _analytics_handle = display(_analytics_box, display_id=True)
    else:
        _analytics_handle = display(_analytics_box, display_id=True)
    _analytics_refresh_impl()
